In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/FeatherViT-Emotion-XXS"
PROJECT_DIR = DRIVE_ROOT

DATASET_NAME = "dog_emotion_384_only_224"  # or "pets_emotion"
DATASET_ROOT = f"/content/drive/MyDrive/datasets/{DATASET_NAME}"

RUNS_ROOT = f"{DRIVE_ROOT}/runs/feathervit_emotion_xxs_{DATASET_NAME}"
TRAIN_DIR = f"{DATASET_ROOT}/train"
VAL_DIR = f"{DATASET_ROOT}/valid"
TEST_DIR = f"{DATASET_ROOT}/test"

OUT_DIR = f"{RUNS_ROOT}/scratch_run_01"
EXPORT_DIR = f"{OUT_DIR}/exports"

ACTIVE_PROJECT_DIR = PROJECT_DIR
ACTIVE_TRAIN_DIR = TRAIN_DIR
ACTIVE_VAL_DIR = VAL_DIR
ACTIVE_TEST_DIR = TEST_DIR
ACTIVE_OUT_DIR = OUT_DIR
ACTIVE_EXPORT_DIR = EXPORT_DIR

In [ ]:
LOCAL_PROJECT_DIR = "/content/FeatherViT-Emotion-XXS"
LOCAL_RUNS_ROOT = "/content/runs/feathervit_emotion_xxs"
LOCAL_OUT_DIR = f"{LOCAL_RUNS_ROOT}/scratch_run_01"
LOCAL_EXPORT_DIR = f"{LOCAL_OUT_DIR}/exports"

In [ ]:
!mkdir -p /content/runs
!rsync -a --delete "$PROJECT_DIR/" "$LOCAL_PROJECT_DIR/"

In [ ]:
ACTIVE_PROJECT_DIR = LOCAL_PROJECT_DIR
ACTIVE_TRAIN_DIR = TRAIN_DIR
ACTIVE_VAL_DIR = VAL_DIR
ACTIVE_TEST_DIR = TEST_DIR
ACTIVE_OUT_DIR = LOCAL_OUT_DIR
ACTIVE_EXPORT_DIR = LOCAL_EXPORT_DIR

In [ ]:
%cd /content
%cd "$ACTIVE_PROJECT_DIR"
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

/content
/content/FeatherViT-Emotion-XXS


In [ ]:
!python -m feathervit_emotion.train \
  --train-dir "$ACTIVE_TRAIN_DIR" \
  --val-dir "$ACTIVE_VAL_DIR" \
  --output-dir "$ACTIVE_OUT_DIR" \
  --epochs 300 \
  --batch-size 64 \
  --num-workers 2 \
  --img-size 224 \
  --lr 5e-4 \
  --weight-decay 0.05 \
  --label-smoothing 0.1 \
  --dropout 0.2 \
  --seed 42 \
  --save-every 10 \
  --val-every 1 \
  --device cuda \
  --amp

Using device: cuda
Model params: 0.95M

Epoch 1/300
train:   0% 0/18 [00:00<?, ?it/s]/content/FeatherViT-Emotion-XXS/feathervit_emotion/train.py:45: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_ctx():
eval:   0% 0/3 [00:00<?, ?it/s]/content/FeatherViT-Emotion-XXS/feathervit_emotion/train.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp_ctx():
train_loss=1.3056 train_top1=39.06 val_loss=1.2842 val_top1=31.25 val_top4=100.00
New best checkpoint saved (top1=31.25)

Epoch 2/300
train_loss=1.2671 train_top1=41.75 val_loss=1.2506 val_top1=31.25 val_top4=100.00

Epoch 3/300
train_loss=1.2476 train_top1=39.58 val_loss=1.2293 val_top1=44.79 val_top4=100.00
New best checkpoint saved (top1=44.79)

Epoch 4/300
train_loss=1.2560 train_top1=42.36 val_loss=1.1852 val_top1=49.90 val_top4=100.00
New best checkpoint saved (to

In [ ]:
!python -m feathervit_emotion.evaluate \
  --val-dir "$ACTIVE_VAL_DIR" \
  --checkpoint "$ACTIVE_OUT_DIR/best.pt" \
  --batch-size 64 \
  --img-size 224 \
  --device cuda

Using device: cuda
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
eval:   0% 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_ration

In [ ]:
SAMPLE_IMAGE = f"{DATASET_ROOT}/test/sad/sad_00002.jpg"

In [ ]:
!python -m feathervit_emotion.predict \
  --image "$SAMPLE_IMAGE" \
  --checkpoint "$ACTIVE_OUT_DIR/best.pt" \
  --img-size 224 \
  --topk 4 \
  --device cuda

Using device: cuda
1. sad: 0.9491
2. angry: 0.0224
3. relaxed: 0.0144
4. happy: 0.0142


In [ ]:
!python -m feathervit_emotion.export \
  --checkpoint "$ACTIVE_OUT_DIR/best.pt" \
  --output-dir "$ACTIVE_EXPORT_DIR" \
  --img-size 224

/content/FeatherViT-Emotion-XXS/feathervit_emotion/model.py:149: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if new_h != h or new_w != w:
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/FeatherViT-Emotion-XXS/feathervit_emotion/export.py", line 54, in <module>
    main()
  File "/content/FeatherViT-Emotion-XXS/feathervit_emotion/export.py", line 35, in main
    traced = torch.jit.trace(model, example)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.py", line 1016, in trace
    traced_func = _trace_impl(
                  ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/jit/_trace.p